# PA-S1 · Экономика привлечения и удержание Яндекс.Афиши
**Продуктовая аналитика · индивидуальный трек Б · осень 2026**

Исследовательский вопрос: какие источники приводят пользователей, которые возвращаются и приносят
достаточно выручки, чтобы покрыть привлечение? Анализ учебной исторической выгрузки, а не оценка современного сервиса.

Ноутбук содержит код расчётов, графики и интерпретации, получаемые при запуске. Для пересчёта: **Restart Kernel → Run All**.
В Google Colab: загрузить этот `.ipynb` через **Файл → Загрузить блокнот**, затем **Выполнить всё**.
Данные автоматически загружаются по HTTPS без аккаунта и проверяются по SHA-256. Интернет нужен только при первой загрузке.
Используются `pandas`, `numpy`, `matplotlib`; локальные версии зафиксированы в `requirements.txt`.

**Навигация:** 1. Продукт → 2. Данные и качество → 3. Определения → 4. Пирамида метрик →
5. Трекинг → 6. Активность и конверсия → 7. Retention → 8. LTV/CAC → 9. Отток → 10. Этика → 11. Цели и выводы.

**Публикация:** ссылка на личный GitHub-репозиторий добавляется после загрузки проекта; инструкция в `README.md`. Перед сдачей укажите своё ФИО и группу и проверьте все выводы самостоятельно. При использовании AI соблюдайте правила курса и укажите помощь в подготовке кода.

## 1. Продукт и гипотеза ценности

**Продукт:** сервис поиска культурных событий и покупки билетов. **Аудитория:** люди, выбирающие
досуг и покупающие билеты онлайн с desktop или touch. **Проблема:** найти подходящее событие
и быстро завершить покупку. **Ценность:** единый каталог и удобное оформление заказа.

**Бизнес-модель для учебного проекта:** доход платформы от продаж/сервисного вознаграждения;
поле `Revenue` используем как доход в единицах выгрузки. Не считаем его GMV, чистой прибылью
или суммой в рублях: валюта и детализация удержанных расходов в CSV не заданы.

**Гипотеза:** каналы с лучшим удержанием на одинаковом горизонте могут иметь более высокую
выручку на привлечённого пользователя. Решение — выбрать кандидатов для контролируемого теста
перераспределения бюджета. Наблюдательные различия каналов сами по себе не доказывают эффект рекламы.

## 2. Паспорт данных и воспроизводимость

Выбран **Yandex Afisha, visits / orders / costs**, период июнь 2017 — май 2018.
У него есть связанные пользовательские визиты, покупки и фактические расходы на рекламу,
поэтому CAC не приходится генерировать или восстанавливать из выручки.

Первичные CSV: [визиты](https://code.s3.yandex.net/datasets/visits_log.csv),
[заказы](https://code.s3.yandex.net/datasets/orders_log.csv),
[расходы](https://code.s3.yandex.net/datasets/costs.csv).
Публичное [зеркало Kaggle, версия 1](https://www.kaggle.com/datasets/slamethariyadi/y-afisha-site-visit)
на дату проверки 29.10.2026 заявляет Apache 2.0. Это заявление загрузившего зеркало,
а не подтверждение прав на исходные данные. Поэтому исходные CSV остаются локально в `data/raw/`,
исключены из git и скачиваются из первичного источника. Версия реально использованных байтов
зафиксирована SHA-256 в этой ячейке и `data/manifest.json`; ETag не принимается за хеш содержимого.

| Таблица | Единица наблюдения | Поля |
|---|---|---|
| visits_log | Визит | Uid, Start Ts, End Ts, Device, Source Id |
| orders_log | Запись о заказе | Uid, Buy Ts, Revenue |
| costs | Расход источника за день | source_id, dt, costs |

**Ограничения выбора:** нет даты регистрации, себестоимости, гео, признака тестового пользователя,
возвратов, согласий и order_id; первое появление — первый наблюдаемый визит. Канал и устройство
дают два исходных измерения. Реальная доля дефектов измеряется, а не доводится искусственно до 5%.
Полный маржинальный lifetime LTV не идентифицируется: считаем revenue LTV за 90 дней и сценарии маржи.
Это отклонение нужно согласовать с преподавателем согласно примечанию PA-S1.
**Google Colab:** запустите все ячейки по порядку. Рабочий каталог — текущая папка Colab, данные автоматически сохраняются в `data/raw/`, а графики в `outputs/`. При перезапуске временной среды Colab данные скачиваются повторно.


In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import URLError
import ssl
import certifi
import hashlib
import json
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display, Markdown

plt.rcParams.update({
    'figure.figsize': (12, 4.5), 'figure.dpi': 125, 'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
    'grid.alpha': 0.18, 'axes.axisbelow': True, 'axes.titleweight': 'bold',
    'axes.titlesize': 13, 'axes.labelsize': 10, 'savefig.bbox': 'tight'
})
BLUE, TEAL, ORANGE, RED = '#2563eb', '#0d9488', '#f59e0b', '#dc2626'
pd.set_option('display.max_columns', 18)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
DATA_DIR = Path('data/raw')
OUT_DIR = Path('outputs')
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)
START = pd.Timestamp('2017-06-01')
END = pd.Timestamp('2018-06-01')  # исключающая правая граница
HORIZON = 90
CHURN_DAYS = 30
print(f'Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}')

In [ ]:
MANIFEST = {'snapshot_date': '2026-10-29', 'source': 'Yandex Practicum public educational files', 'kaggle_mirror': 'https://www.kaggle.com/datasets/slamethariyadi/y-afisha-site-visit', 'kaggle_mirror_version': 1, 'license_note': 'Kaggle mirror declares Apache 2.0; upstream redistribution license not confirmed. Raw files are excluded from git.', 'files': {'visits_log.csv': {'url': 'https://code.s3.yandex.net/datasets/visits_log.csv', 'sha256': '421674ac4cdac63b34ce87e13b057083f3f2fb8bbd56439763acfc8de36265d2', 'bytes': 25117663}, 'orders_log.csv': {'url': 'https://code.s3.yandex.net/datasets/orders_log.csv', 'sha256': 'd06042bc76612758ea48900eb471f1b249ecaf16e5047dbcd80862e440a9ac86', 'bytes': 2287508}, 'costs.csv': {'url': 'https://code.s3.yandex.net/datasets/costs.csv', 'sha256': 'f6e47f518201ea38e2cab930b8759e4afbdda36e18325ca2cb3832eacaeaa639', 'bytes': 49258}}}

for name, spec in MANIFEST['files'].items():
    path = DATA_DIR / name
    if not path.exists():
        try:
            request = Request(spec['url'], headers={'User-Agent': 'PA-S1 educational analysis'})
            with urlopen(request, timeout=120, context=ssl.create_default_context(cafile=certifi.where())) as response:
                content = response.read()
        except (URLError, TimeoutError) as exc:
            raise RuntimeError(f'Не удалось скачать {name}. Проверьте интернет или загрузите CSV в data/raw/.') from exc
        if hashlib.sha256(content).hexdigest() != spec['sha256']:
            raise ValueError(f'Источник {name} изменился: требуется проверка новой версии данных.')
        path.write_bytes(content)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != spec['sha256']:
        raise ValueError(f'Хеш {name} не совпадает со снимком; файл не использован.')
    print(f'{name}: SHA-256 OK, {path.stat().st_size:,} байт')

# Uid больше 2**53: чтение через float разрушило бы идентификаторы.
raw_visits = pd.read_csv(DATA_DIR / 'visits_log.csv', dtype={'Uid': 'string'})
raw_orders = pd.read_csv(DATA_DIR / 'orders_log.csv', dtype={'Uid': 'string'})
raw_costs = pd.read_csv(DATA_DIR / 'costs.csv')
display(pd.DataFrame([
    {'Таблица': name, 'Строк': len(frame), 'Столбцов': frame.shape[1], 'Пропусков': int(frame.isna().sum().sum()),
     'Полных дублей': int(frame.duplicated().sum())}
    for name, frame in [('visits', raw_visits), ('orders', raw_orders), ('costs', raw_costs)]
]))

### 2.1. Правила качества до расчётов

Основное окно **[01.06.2017, 01.06.2018)**. Визит относится к дню начала; конец сессии
может быть после полуночи. Визит с ошибочной длительностью остаётся свидетельством активности,
но его длительность не используется. Нулевая выручка не является платной покупкой.
Точные дубли визитов/расходов удалялись бы после отчёта; строки заказов без order_id автоматически
не дедуплицируем: два одинаковых заказа могут быть легитимными. В этой версии точных дублей нет.
Экстремальные положительные суммы сохраняем, отдельно показывая чувствительность к верхнему 1%.

In [ ]:
visits = raw_visits.rename(columns={'Uid':'user_id', 'Start Ts':'start', 'End Ts':'end',
                                  'Source Id':'source', 'Device':'device'}).copy()
orders = raw_orders.rename(columns={'Uid':'user_id', 'Buy Ts':'buy', 'Revenue':'revenue'}).copy()
costs = raw_costs.rename(columns={'source_id':'source', 'dt':'date', 'costs':'spend'}).copy()
for col in ['start', 'end']:
    visits[col] = pd.to_datetime(visits[col], errors='coerce')
orders['buy'] = pd.to_datetime(orders['buy'], errors='coerce')
costs['date'] = pd.to_datetime(costs['date'], errors='coerce')
visits['duration_min'] = (visits['end'] - visits['start']).dt.total_seconds() / 60
q99 = orders['revenue'].quantile(.99)
quality = pd.DataFrame([
    ('Не удалось разобрать время начала визита', visits.start.isna().sum(), len(visits)),
    ('Не удалось разобрать время заказа', orders.buy.isna().sum(), len(orders)),
    ('Конец визита раньше начала', (visits.duration_min < 0).sum(), len(visits)),
    ('Визит нулевой длительности', (visits.duration_min == 0).sum(), len(visits)),
    ('Визит длиннее 4 часов: флаг проверки', (visits.duration_min > 240).sum(), len(visits)),
    ('Заказ с нулевой выручкой', (orders.revenue == 0).sum(), len(orders)),
    ('Заказ с отрицательной выручкой', (orders.revenue < 0).sum(), len(orders)),
    ('Заказ выше P99: флаг, не доказанная ошибка', (orders.revenue > q99).sum(), len(orders)),
    ('Заказ вне основного окна', (~orders.buy.between(START, END, inclusive='left')).sum(), len(orders)),
], columns=['Проверка', 'Строк', 'Всего'])
quality['Доля, %'] = quality['Строк'] / quality['Всего'] * 100
display(quality)
display(pd.DataFrame([
    ('visits.start', visits.start.min(), visits.start.max()),
    ('visits.end', visits.end.min(), visits.end.max()),
    ('orders.buy', orders.buy.min(), orders.buy.max()),
    ('costs.date', costs.date.min(), costs.date.max()),
], columns=['Поле', 'Минимум', 'Максимум']))

visits = visits.loc[visits.start.ge(START) & visits.start.lt(END) & visits.user_id.notna()].drop_duplicates().copy()
orders = orders.loc[orders.buy.ge(START) & orders.buy.lt(END) & orders.user_id.notna() & orders.revenue.ge(0)].copy()
costs = costs.loc[costs.date.ge(START) & costs.date.lt(END)].drop_duplicates().copy()
assert visits.start.notna().all() and orders.buy.notna().all()
assert costs.spend.ge(0).all() and not costs.duplicated(['source', 'date']).any()
visits.loc[visits.duration_min.lt(0), 'duration_min'] = np.nan
paid = orders.loc[orders.revenue.gt(0)].copy()
print(f'В анализе: {len(visits):,} визитов; {len(orders):,} записей заказов; {len(paid):,} платных заказов.')

In [ ]:
# Детерминированное разрешение одинакового времени: source, затем device.
first_visits = visits.sort_values(['start', 'source', 'device'], kind='stable').drop_duplicates('user_id')
users = first_visits[['user_id','start','source','device']].rename(columns={'start':'first_seen'}).copy()
users['cohort'] = users.first_seen.dt.to_period('M')
orders = orders.merge(users[['user_id','first_seen','cohort','source','device']], on='user_id', how='left', validate='many_to_one')
paid = orders.loc[orders.revenue.gt(0)].copy()
assert orders.first_seen.notna().all(), 'Заказы без пользовательского профиля'
assert orders.buy.ge(orders.first_seen).all(), 'Покупка раньше первого визита'
paid['age_days'] = (paid.buy - paid.first_seen).dt.total_seconds() / 86400
visits['date'] = visits.start.dt.normalize()
paid['date'] = paid.buy.dt.normalize()
costs['cohort'] = costs.date.dt.to_period('M')
calendar = pd.date_range(START, END - pd.Timedelta(days=1), freq='D')
spend_sources = set(costs.source.unique())
print('Каналы визитов:', sorted(visits.source.unique().tolist()))
print('Каналы с расходами:', sorted(spend_sources))
print('Устройства:', sorted(visits.device.unique().tolist()))
print('Покупателей без визитов: 0; покупок до первого визита: 0.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
positive = paid.revenue
axes[0].hist(np.log10(positive), bins=60, color=BLUE, alpha=.85)
axes[0].set(title='Распределение выручки заказа', xlabel='log10(выручка, ед. выгрузки)', ylabel='Платных заказов')
dur = visits.duration_min.dropna()
axes[1].hist(dur.clip(upper=60), bins=50, color=TEAL, alpha=.85)
axes[1].set(title='Длительность визита', xlabel='Минуты (значения >60 в последнем столбце)', ylabel='Визитов')
plt.tight_layout(); plt.savefig(OUT_DIR / '01_quality.png'); plt.show()
top1_share = positive[positive > positive.quantile(.99)].sum() / positive.sum()
display(Markdown(f'''**Качество и ограничения.** Пропусков и точных дублей в исходных CSV нет;
это не означает отсутствия скрытых дефектов. Нулевые длительности могут отражать округление времени
или уход без второго события, поэтому их не удаляем. Верхний 1% сумм формирует **{top1_share:.1%}**
выручки: среднее чувствительно к крупным заказам. Тестовые пользователи достоверно не определяются:
нет ни флага, ни списка сотрудников; высокая активность сама по себе не основание удалить человека.
Часовой пояс не указан: сохраняем исходное время без выдуманной конвертации в UTC.'''))

## 3. Контракт определений · версия 1

| Понятие | Зафиксированное определение |
|---|---|
| Пользователь | Строковый `Uid`; нет гарантии, что один UID равен одному человеку |
| Появление | Первый наблюдаемый `Start Ts`; это не дата регистрации и не доказанное первое знакомство |
| Активность | Хотя бы один старт визита; заказ без нового визита не создаёт активность задним числом |
| DAU / WAU / MAU | Уникальные активные за день / календарную неделю Пн–Вс / календарный месяц |
| Конверсия C30 | Доля новых пользователей с положительным заказом в `[first_seen, first_seen+30 суток)`; только полностью наблюдаемые окна |
| Когорта | Календарный месяц первого наблюдаемого визита |
| M1 retention | Доля когорты с визитом в следующем календарном месяце; M0=100%; это не D30 и не rolling retention |
| R7–13 | Визит в `[first_seen+7 суток, first_seen+14 суток)`; полностью наблюдаемые окна |
| Отток на срезе | Последний визит строго раньше `END−30 суток`: за последние 30 дней нет ни одного старта визита |
| CAC_user | Рекламные расходы канала/месяца ÷ новые наблюдаемые пользователи того же канала/месяца; знаменатель включает непокупателей |
| Revenue LTV90 | Сумма положительной выручки в `[first_seen, first_seen+90 суток)` ÷ **все** привлечённые пользователи зрелой когорты |
| Сценарный LTV90(m) | Revenue LTV90 × заданная маржа m; это вклад до расходов привлечения, не фактическая прибыль |
| LTV90/CAC | Для одинаковых канала, когорты и знаменателя: выручка90 × m ÷ расходы привлечения |
| Атрибуция | First touch: канал и устройство первого визита. Они фиксированы для пользователя, даже если позже изменились |

Неполные месяцы retention маскируются `NaN`, а не нулём. Для LTV90 берём **целые месячные когорты**,
у каждого участника которых полностью закончились 90 суток. Вклад старых до начала лога пользователей
не отделим; расходы на ретаргетинг не выделены. Поэтому CAC — наблюдательная оценка, а не причинная стоимость инкрементального клиента.

## 4. Пирамида метрик: от ценности к решению

| Уровень | Метрика | Почему выбрана | Решение |
|---|---|---|---|
| North Star | Уникальные покупатели с положительным заказом за календарный месяц | Измеряет полученную покупателями ценность, не число визитов | Планировать рост покупающей аудитории |
| Диагностика 1 | C30 | Связывает привлечение с первой покупкой | Проверять посадочные страницы и оформление заказа |
| Диагностика 2 | M1 retention | Показывает возврат после первого месяца | Проверять подборки и напоминания о событиях |
| Риск / guardrail | CAC / revenue LTV90 по зрелым платным когортам | Даже при марже 100% значение >1 означает нехватку выручки90 | Не масштабировать канал до проверки экономики |

При подтверждённой марже guardrail заменяется на CAC / LTV90(m). Порог 1 означает покрытие
только рекламных расходов за 90 дней; зарплаты и фиксированные расходы сюда не входят.
DAU, WAU, MAU, R7–13 и отток — вспомогательные показатели интерпретации.

## 5. План трекинга

Три наблюдаемых типа события (`session_start`, `session_end`, `order_recorded`) удовлетворяют треку Б. Производные события
не выдаются за дополнительные сырые действия. `checkout_start` и показы карточек отсутствуют:
подробную воронку оформления на этих данных восстановить нельзя.

| Событие | Триггер | Параметры | Метрика | Этап пути | Поля датасета |
|---|---|---|---|---|---|
| session_start | Начался визит | user_id, timestamp, source, device | DAU/WAU/MAU, retention, C30: знаменатель | Вход / возврат | visits: Uid, Start Ts, Source Id, Device |
| session_end | Визит завершён | user_id, start, end, duration | Контроль качества визитов, интерпретация активности | Использование | visits: Uid, Start Ts, End Ts |
| order_recorded | Появилась запись заказа | user_id, timestamp, revenue | North Star, C30, LTV90 при revenue>0 | Монетизация | orders: Uid, Buy Ts, Revenue |
| first_seen (производное) | Первый session_start UID | first_seen, first_source, first_device | Когорта, CAC_user, C30 | Привлечение | min(visits.Start Ts), Uid, Source Id, Device |
| marketing_spend (агрегат) | Закрылся день расходов | date, source, spend | CAC_user, LTV90/CAC | Привлечение | costs: dt, source_id, costs |

Для реального внедрения нужны уникальные event_id / order_id, серверное время, версия схемы,
разделение сотрудников/ботов и согласий. Это план будущего трекинга, не существующие поля CSV.

## 6. Активность, North Star и конверсия

Календарную неделю с неполным наблюдением исключаем из среднего WAU. Для каждого дня
строим непрерывную шкалу; отсутствие строк показано как 0, но причина таких провалов требует проверки.

In [ ]:
dau = visits.groupby('date').user_id.nunique().reindex(calendar, fill_value=0)
week = visits.start.dt.to_period('W-SUN')
wau_all = visits.groupby(week).user_id.nunique()
full_weeks = (wau_all.index.start_time >= START) & (wau_all.index.end_time < END)
wau = wau_all.loc[full_weeks]
mau = visits.groupby(visits.start.dt.to_period('M')).user_id.nunique()
north_star = paid.groupby(paid.buy.dt.to_period('M')).user_id.nunique().reindex(mau.index, fill_value=0)
daily_revenue = paid.groupby('date').revenue.sum().reindex(calendar, fill_value=0)
overview = pd.DataFrame({
    'Показатель': ['Наблюдаемые пользователи','Платные покупатели','Платные заказы','Выручка','Расходы',
                   'Средний DAU','Средний WAU (полные недели)','Средний MAU','Средний платный чек','Медианный платный чек'],
    'Значение': [len(users),paid.user_id.nunique(),len(paid),paid.revenue.sum(),costs.spend.sum(),
                 dau.mean(),wau.mean(),mau.mean(),paid.revenue.mean(),paid.revenue.median()]
})
display(overview)
display(pd.DataFrame({'MAU':mau,'North Star: покупатели':north_star,
                      'Доля покупателей среди MAU':north_star/mau}))
fig, axes = plt.subplots(2, 1, figsize=(12, 7), constrained_layout=True)
axes[0].plot(dau.index, dau, alpha=.25, color=BLUE, label='DAU')
axes[0].plot(dau.index, dau.rolling(7, min_periods=7).mean(), color=BLUE, lw=2, label='Среднее за 7 дней')
axes[0].set(title='Активность: ежедневный ряд и сглаживание', ylabel='Пользователей'); axes[0].legend()
axes[1].bar(north_star.index.astype(str), north_star, color=TEAL)
axes[1].tick_params(axis='x', rotation=45)
axes[1].set(title='North Star: уникальные платные покупатели', ylabel='Покупателей в месяц')
plt.savefig(OUT_DIR / '02_activity.png'); plt.show()
display(Markdown(f'''**Наблюдение.** Средний DAU — **{dau.mean():,.0f}**, MAU — **{mau.mean():,.0f}**;
North Star максимальна в **{north_star.idxmax()}** ({north_star.max():,} покупателей),
минимальна в **{north_star.idxmin()}** ({north_star.min():,}). Изменения объёма не доказывают улучшение продукта:
сначала нужно отделить календарную сезонность и изменение рекламы. Дней без визитов: **{int(dau.eq(0).sum())}**.'''))

In [ ]:
first_paid = paid.groupby('user_id').buy.min().rename('first_paid')
user_metrics = users.join(first_paid, on='user_id')
user_metrics['converted30'] = ((user_metrics.first_paid - user_metrics.first_seen).dt.total_seconds() / 86400).between(0,30,inclusive='left')
eligible30 = user_metrics.first_seen.add(pd.Timedelta(days=30)).le(END)
conversion = user_metrics.loc[eligible30].groupby('source').agg(n=('user_id','size'), buyers30=('converted30','sum'))
conversion['C30'] = conversion.buyers30 / conversion.n

# Wilson 95% CI: описательная неопределённость биномиальной доли, не причинный эффект.
def wilson(successes, n, z=1.96):
    p = successes / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half = z * np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / denom
    return center-half, center+half

conversion['lo'], conversion['hi'] = wilson(conversion.buyers30, conversion.n)
display(conversion)
shown = conversion.loc[conversion.n.ge(100)].sort_values('C30')
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(shown.index.astype(str), shown.C30, color=BLUE)
ax.errorbar(shown.C30, np.arange(len(shown)), xerr=[shown.C30-shown.lo,shown.hi-shown.C30],
            fmt='none', ecolor='#1e293b', capsize=3)
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.set(title='C30 по first-touch каналу · 95% интервалы Wilson · n ≥ 100', xlabel='Конверсия за 30 суток', ylabel='Канал')
plt.tight_layout(); plt.savefig(OUT_DIR / '03_conversion.png'); plt.show()
c30_overall = user_metrics.loc[eligible30, 'converted30'].mean()
display(Markdown(f'''**Наблюдение.** Общая C30 — **{c30_overall:.2%}** на **{eligible30.sum():,}** пользователях
с полным окном. Каналы 6/7 остаются в таблице, но малые группы не ранжируются на графике.
Различия могут объясняться аудиторией и сезонностью; для выбора изменений нужен эксперимент.'''))

## 7. Когортное удержание

Считаем уникальных вернувшихся, а не число сессий. Взвешенное M1 — сумма вернувшихся
по зрелым когортам / сумма размеров тех же когорт. Оно отличается от среднего процентов по строкам.
Для сравнения источников дополнительно используем одинаковое персональное окно R7–13.

In [ ]:
activity = visits[['user_id','start']].merge(users[['user_id','cohort','first_seen','source','device']], on='user_id', validate='many_to_one')
activity['active_month'] = activity.start.dt.to_period('M')
activity['age_month'] = activity.active_month.astype('int64') - activity.cohort.astype('int64')
cohort_sizes = users.groupby('cohort').size()
cohorts = pd.period_range(START, END-pd.Timedelta(days=1), freq='M')
counts = activity.groupby(['cohort','age_month']).user_id.nunique().unstack().reindex(index=cohorts, columns=range(len(cohorts)))
retention = counts.fillna(0).div(cohort_sizes, axis=0)
for cohort in retention.index:
    for age in retention.columns:
        if (cohort + int(age) + 1).start_time > END:
            retention.loc[cohort, age] = np.nan
assert np.allclose(retention[0],1)
assert retention.stack().between(0,1).all()
display(retention.style.format('{:.1%}', na_rep='—').background_gradient(cmap='Blues', axis=None, subset=retention.columns[1:]))
fig, ax = plt.subplots(figsize=(13, 6))
matrix = retention.to_numpy().copy()
matrix[:,0] = np.nan
im = ax.imshow(matrix, cmap='Blues', vmin=0, vmax=.15, aspect='auto')
ax.set_xticks(range(len(retention.columns)), [f'M{x}' for x in retention.columns])
ax.set_yticks(range(len(retention)), [f'{c} · n={cohort_sizes[c]:,}' for c in retention.index])
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        if j == 0:
            ax.text(j,i,'100%',ha='center',va='center',color='#64748b',fontsize=9)
        elif np.isfinite(matrix[i,j]):
            ax.text(j,i,f'{matrix[i,j]:.1%}',ha='center',va='center',fontsize=9,
                    color='white' if matrix[i,j]>.085 else '#0f172a')
ax.set(title='Календарный retention · белые ячейки справа ещё не наблюдались', xlabel='Месяц жизни', ylabel='Когорта первого визита')
fig.colorbar(im, ax=ax, format=PercentFormatter(1), label='Доля когорты; шкала M1+')
plt.tight_layout(); plt.savefig(OUT_DIR / '04_retention.png'); plt.show()
mature_m1 = retention[1].notna()
weighted_m1 = (retention.loc[mature_m1,1]*cohort_sizes[mature_m1]).sum()/cohort_sizes[mature_m1].sum()
display(Markdown(f'''**Вывод.** Взвешенное M1 составляет **{weighted_m1:.2%}**.
Нулевое значение в наблюдённой ячейке означает отсутствие возвратов; пустая ячейка — недостаток времени наблюдения.
Первый месяц неполон на персональном уровне, поэтому M1 не заменяет фиксированное окно D30.
Низкий возврат даёт основание тестировать повторный интерес к событиям, но не доказывает причину ухода.'''))
retention.to_csv(OUT_DIR / 'retention_monthly.csv')

In [ ]:
activity['age_days'] = (activity.start-activity.first_seen).dt.total_seconds()/86400
returned_ids = activity.loc[activity.age_days.between(7,14,inclusive='left'),'user_id'].unique()
fixed_retention = users.loc[users.first_seen.add(pd.Timedelta(days=14)).le(END)].copy()
fixed_retention['returned'] = fixed_retention.user_id.isin(returned_ids)
segment_tables = {}
fig, axes = plt.subplots(1,2,figsize=(12,4))
for ax, dim, label in zip(axes,['source','device'],['Канал первого визита','Устройство первого визита']):
    table = fixed_retention.groupby(dim).agg(n=('user_id','size'), returned=('returned','sum'))
    table['R7–13'] = table.returned/table.n
    table['lo'],table['hi'] = wilson(table.returned,table.n)
    segment_tables[dim] = table
    show = table.loc[table.n.ge(100)].sort_values('R7–13')
    ax.bar(show.index.astype(str),show['R7–13'],color=TEAL)
    ax.errorbar(np.arange(len(show)),show['R7–13'],yerr=[show['R7–13']-show.lo,show.hi-show['R7–13']],fmt='none',ecolor='#334155',capsize=3)
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.set(title=label, ylabel='R7–13 · 95% CI · n ≥ 100')
plt.tight_layout(); plt.savefig(OUT_DIR / '05_retention_segments.png'); plt.show()
display(segment_tables['source']); display(segment_tables['device'])
display(Markdown('**Интерпретация.** Это сравнение first-touch сегментов на равном горизонте. Интервалы не учитывают смешение когорт и устройств; связь канала с возвратом не равна причинному эффекту канала.'))

## 8. LTV90 / CAC: единый знаменатель и зрелость

Для когорты месяца c и источника s:

```text
CAC(c,s)        = Spend(c,s) / N(c,s)
Revenue LTV90  = Revenue(c,s, возраст от 0 до 90 суток) / N(c,s)
LTV90(m) / CAC = m × Revenue90(c,s) / Spend(c,s)
```

Здесь N включает **всех** новых наблюдаемых пользователей, в том числе без покупки.
Расходы — весь бюджет канала в месяце появления этой когорты. Это приближение к привлечению:
в расходах могут быть ретаргетинг и работа с ранее существовавшей аудиторией.

Суммы и пользователи агрегируются **до деления**, среднее отношений каналов не используется.
Источник без строки расходов не считается бесплатным: для него CAC неизвестен.
Горизонт 90 дней — наблюдённая накопленная выручка, не прогноз всего срока жизни.
Маржи 20%, 50%, 80%, 100% — открыто заданные сценарии; 100% является верхней границей.
**Важно:** это наблюдаемый 90-дневный *revenue LTV*, а не пожизненная прибыль. Без сведений о марже нельзя утверждать, что канал окупается; коэффициент при 100% марже — верхняя оценка.


In [ ]:
eligible_cohorts = [c for c in cohort_sizes.index if (c+1).start_time+pd.Timedelta(days=HORIZON) <= END]
acquired = users.loc[users.cohort.isin(eligible_cohorts)]
within90 = paid.loc[paid.cohort.isin(eligible_cohorts) & paid.age_days.between(0,HORIZON,inclusive='left')]
n = acquired.groupby(['cohort','source']).size().rename('users')
r90 = within90.groupby(['cohort','source']).revenue.sum().rename('revenue90')
s = costs.loc[costs.cohort.isin(eligible_cohorts)].groupby(['cohort','source']).spend.sum().rename('spend')
economics = pd.concat([n,r90,s],axis=1)
assert economics.users.notna().all(), 'Есть рекламные расходы без наблюдаемых новых пользователей'
economics['revenue90'] = economics.revenue90.fillna(0)
economics['LTV90_rev'] = economics.revenue90/economics.users
economics['CAC_user'] = economics.spend/economics.users
economics['ratio_rev'] = economics.revenue90/economics.spend
economics['ratio_m50'] = .5*economics.ratio_rev
known = economics.loc[economics.spend.notna() & economics.spend.gt(0)].copy()
channel_economics = known.groupby('source')[['users','revenue90','spend']].sum()
channel_economics['LTV90_rev'] = channel_economics.revenue90/channel_economics.users
channel_economics['CAC_user'] = channel_economics.spend/channel_economics.users
channel_economics['ratio_rev'] = channel_economics.revenue90/channel_economics.spend
channel_economics['ratio_m50'] = .5*channel_economics.ratio_rev
channel_economics['break_even_margin'] = channel_economics.spend/channel_economics.revenue90
display(channel_economics)
print('Зрелые месячные когорты:', ', '.join(map(str,eligible_cohorts)))
print('Пользователей зрелых когорт без известных расходов:', int(economics.loc[economics.spend.isna(),'users'].sum()))
total_users,total_rev90,total_spend = known[['users','revenue90','spend']].sum()
pooled_ltv90,pooled_cac = total_rev90/total_users,total_spend/total_users
pooled_ratio = total_rev90/total_spend
display(Markdown(f'''**Общая оценка зрелых платных когорт:** revenue LTV90 = **{pooled_ltv90:.3f}**,
CAC_user = **{pooled_cac:.3f}**, revenue LTV90/CAC = **{pooled_ratio:.3f}**.
При сценарной марже 50% LTV90/CAC = **{.5*pooled_ratio:.3f}**.
Это результат по одним и тем же когортам и источникам; выручка всего года не делится на CAC другой группы.'''))
economics.to_csv(OUT_DIR / 'economics_cohort_source.csv')
channel_economics.to_csv(OUT_DIR / 'economics_channels.csv')

In [ ]:
ordered = channel_economics.sort_values('ratio_rev')
fig, axes = plt.subplots(1,2,figsize=(13,4.5))
x = np.arange(len(ordered))
axes[0].bar(x-.18,ordered.CAC_user,.36,label='CAC_user',color=ORANGE)
axes[0].bar(x+.18,ordered.LTV90_rev,.36,label='Revenue LTV90',color=BLUE)
axes[0].set_xticks(x, ordered.index.astype(str))
axes[0].set(title='Стоимость и доход на привлечённого пользователя',xlabel='First-touch канал',ylabel='Единицы выгрузки'); axes[0].legend()
axes[1].barh(ordered.index.astype(str),ordered.ratio_rev,color=BLUE,label='Маржа 100%: верхняя граница')
axes[1].barh(ordered.index.astype(str),ordered.ratio_m50,color=TEAL,label='Сценарий маржи 50%')
axes[1].axvline(1,color=RED,ls='--',label='Покрытие рекламы за 90 дней')
axes[1].set(title='LTV90/CAC по одинаково зрелым когортам',xlabel='Отношение'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT_DIR / '06_economics.png'); plt.show()
margins = pd.Series([.2,.5,.8,1.],name='Маржа')
sensitivity = pd.DataFrame({str(s):margins*row.ratio_rev for s,row in channel_economics.iterrows()})
sensitivity.index = [f'{m:.0%}' for m in margins]
sensitivity.index.name='Сценарий маржи'
display(sensitivity.style.format('{:.2f}').background_gradient(cmap='RdYlGn',axis=None,vmin=0,vmax=2))
best_source = channel_economics.ratio_rev.idxmax()
best_ratio = channel_economics.loc[best_source,'ratio_rev']
be = channel_economics.loc[best_source,'break_even_margin']
display(Markdown(f'''**Вывод.** Наибольшее отношение revenue LTV90/CAC — у канала **{best_source}**:
**{best_ratio:.2f}**; для покрытия рекламы на этом горизонте нужна маржа не ниже **{be:.1%}**.
Порог выше 100% недостижим даже при отсутствии себестоимости. При марже 50% покрывают рекламу
**{int(channel_economics.ratio_m50.ge(1).sum())} из {len(channel_economics)}** каналов.
Это кандидаты для проверки, а не автоматическая рекомендация повысить бюджет: атрибуция и инкрементальность не установлены.'''))

In [ ]:
cap = within90.revenue.quantile(.99)
rev_by_source = within90.groupby('source').revenue.sum()
capped = within90.assign(revenue_capped=within90.revenue.clip(upper=cap)).groupby('source').revenue_capped.sum()
robustness = channel_economics[['ratio_rev']].copy()
robustness['ratio_rev_capped_P99'] = capped.reindex(robustness.index)/channel_economics.spend
robustness['change_pct'] = (robustness.ratio_rev_capped_P99/robustness.ratio_rev-1)*100
display(robustness)
display(Markdown('**Чувствительность.** P99-ограничение используется только как стресс-сценарий. В основном расчёте выручка не обрезана. Заметная смена ранжирования потребует проверки крупных покупателей и возможных B2B-заказов; удалять их без причины нельзя.'))

## 9. Отток как наблюдаемая неактивность

Срез — исключающая граница 01.06.2018. Нет визита в последних 30 днях — operational churn,
но не доказанный уход навсегда. Недавно пришедшие пользователи с активностью в окне не отточные;
для обучения модели такие неполные персональные окна потребуется цензурировать отдельно.
Проверяем чувствительность к N=14, 30, 60 дней.

In [ ]:
churn = users.join(last_active.rename('last_active'),on='user_id')
churn['churn30'] = churn.last_active.lt(END-pd.Timedelta(days=CHURN_DAYS))
churn_table = churn.groupby('source').agg(n=('user_id','size'),churned=('churn30','sum'))
churn_table['churn30'] = churn_table.churned/churn_table.n
display(churn_table)
churn_sensitivity = pd.DataFrame([
    {'N, дней':n,'Неактивных':int(last_active.lt(END-pd.Timedelta(days=n)).sum()),
     'Доля':last_active.lt(END-pd.Timedelta(days=n)).mean()} for n in [14,30,60]
])
display(churn_sensitivity)
fig, ax = plt.subplots(figsize=(8,3.8))
ax.bar(churn_sensitivity['N, дней'].astype(str),churn_sensitivity['Доля'],color=ORANGE)
ax.yaxis.set_major_formatter(PercentFormatter(1)); ax.set(title='Как правило неактивности меняет оценку оттока',xlabel='Дней без визита к концу окна',ylabel='Доля всех наблюдаемых пользователей')
plt.tight_layout(); plt.savefig(OUT_DIR / '07_churn.png'); plt.show()
display(Markdown(f'''**Вывод.** По правилу 30 дней неактивны **{churn.churn30.sum():,}** пользователей
({churn.churn30.mean():.1%}). Старые когорты успели провести больше времени без визита,
поэтому сравнивать этот срез как качество каналов нельзя без выравнивания возраста пользователей.
Для билетного продукта разумно отдельно проверять окна 60–90 дней и реактивацию.'''))

## 10. Этика и происхождение

| Пункт | Что известно / действие в проекте |
|---|---|
| Цель | Учебная агрегированная оценка удержания и экономики, без индивидуальных решений о людях |
| Согласие | Записи согласий не приложены: наличие правового основания по CSV подтвердить нельзя |
| Специальные категории | Явных полей нет; покупки и временные паттерны всё равно могут раскрывать интересы |
| Идентификаторы | UID — псевдоним, не доказательство необратимой анонимизации; персональные UID в выводы не публикуются |
| Третьи лица | Не обогащать внешними данными и не пытаться установить личности; передавать только агрегаты |
| Малые группы | Срезы с n<100 не ранжируются на графиках; таблицы содержат агрегированные размеры |
| Хранение | Исходные файлы локально в data/raw, не попадают в git; рабочие CSV удалить после завершения курса |
| Удаление | Удалить data/raw и локальные копии; для удаления одного субъекта нет прямого канала к исходному владельцу |
| Лицензия | Зеркало Kaggle v1 заявляет Apache 2.0, права исходного владельца отдельно не подтверждены; лицензия чужого ноутбука не является лицензией данных |

Остаточные риски — неизвестные согласия, непроверенное происхождение и возможность связать псевдоним
с человеком. Они ограничивают использование учебным анализом; не считаются юридически устранёнными
одной анонимизацией UID. В репозитории остаются код, паспорт источника и агрегированные результаты.

## 11. Критерии успеха к финалу проекта

Цели ниже — **предложения для будущего пилота**, не уже достигнутый эффект. Исторический датасет
не содержит A/B-разбиения и не позволяет честно измерить результат будущего изменения.

In [ ]:
latest_north_star = int(north_star.iloc[-1])
goals = pd.DataFrame([
    ('North Star, покупателей/месяц',latest_north_star,int(np.ceil(latest_north_star*1.10)),
     'Рост ≥10% к последнему полному месяцу; в пилоте — покупатели на 1000 рандомизированных пользователей при равных бюджетах'),
    ('C30',c30_overall,c30_overall+.02,
     '+2 п.п.; A/B по пользователям, полные 30 суток, 95% CI разности и план мощности до старта'),
    ('M1 retention',weighted_m1,weighted_m1+.01,
     '+1 п.п.; A/B, завершённый следующий месяц, одинаковое распределение месяцев привлечения'),
],columns=['Метрика','Исторический baseline','Цель','Проверка'])
display(goals)
display(Markdown('**Guardrail для всех целей:** CAC/revenue LTV90 не выше 1 (эквивалентно revenue LTV90/CAC ≥ 1) на полностью зрелых платных когортах; если baseline хуже, сначала добиться покрытия рекламы. После получения себестоимости пересчитать по марже. Это ограничение пилота, а не обещание достижимости.'))

summary = {
    'users':int(len(users)), 'visits':int(len(visits)), 'paid_orders':int(len(paid)),
    'revenue':float(paid.revenue.sum()), 'spend':float(costs.spend.sum()),
    'mean_DAU':float(dau.mean()), 'C30':float(c30_overall),'weighted_M1':float(weighted_m1),
    'LTV90_revenue_paid_sources':float(pooled_ltv90),'CAC_user_paid_sources':float(pooled_cac),
    'LTV90_revenue_to_CAC':float(pooled_ratio),'scenario_m50_LTV90_to_CAC':float(.5*pooled_ratio),
    'churn30':float(churn.churn30.mean()), 'best_revenue_ratio_source':int(best_source),
    'mature_cohorts':list(map(str,eligible_cohorts))
}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
quality.to_csv(OUT_DIR / 'quality_checks.csv',index=False)
overview.to_csv(OUT_DIR / 'overview.csv',index=False)
conversion.to_csv(OUT_DIR / 'conversion_channels.csv')
goals.to_csv(OUT_DIR / 'success_criteria.csv',index=False)
display(Markdown(f'''### Итог и следующие решения

1. **Удержание:** M1 = **{weighted_m1:.2%}**; R7–13 и срезы показывают, где проверять повторный интерес.
2. **Экономика:** revenue LTV90/CAC = **{pooled_ratio:.2f}** для зрелых платных когорт;
   при марже 50% — **{.5*pooled_ratio:.2f}**. Полный lifetime LTV и фактическая прибыль не измерены.
3. **Привлечение:** канал **{best_source}** — кандидат для проверки по observed revenue LTV90/CAC;
   подтвердить себестоимость, first-touch атрибуцию и инкрементальность перед изменением бюджета.
4. **Данные:** запросить order_id, возвраты, маржу, регистрации и признак тестовых записей.
   Согласовать с преподавателем отсутствие себестоимости и отличающуюся от ориентира долю дефектов.
5. **Сдача:** добавить ФИО/группу и реальную ссылку на репозиторий после его публикации.
   Все вычисления используют относительные пути; таблицы и рисунки сохранены в `outputs/`.
'''))

### Контроль соответствия PA-S1

| Раздел задания | Где выполнен |
|---|---|
| Продукт, аудитория, гипотеза, модель | §1 |
| Источник, версия, лицензия, объём, ограничения | §2, SHA-256, data/manifest.json |
| Пропуски, дубли, выбросы, тестовые пользователи, время | §2.1 |
| Активность, конверсия, retention, когорта, отток | §3 |
| North Star, 2 диагностические, 1 риск-метрика | §4 |
| ≥3 события и связь с полями/метриками | §5 |
| Retention и LTV/CAC, графики и выводы | §6–9 |
| Этика | §10 |
| 2–3 цели и проверка | §11 |
| Запуск и окружение | README.md, requirements.txt, .gitignore |
| Ссылка на репозиторий, ФИО и группа | Требуют заполнения автором после публикации |

Требования: приложенный технический тикет `PA-S1` (осень 2026). Дополнительный документ с требованиями к датасету в этом комплекте не предоставлен.